<a href="https://colab.research.google.com/github/Freibott/clasificador-milito/blob/main/clasificador_milito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Freibott/clasificador-milito/blob/main/clasificador_milito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificador de textos en esperanto: WWI, WWII y Guerra Fría

Minería de Textos · CUNEF · 2026/2027

## Fase 1 · Dataset

### Exploración de fuentes

**Objetivo.** Antes de construir el dataset, comprobamos (27/10/2026) si Vikipedio (Wikipedia en esperanto) tiene texto suficiente para WWI, WWII y Guerra Fría. Para ello consultamos su API de MediaWiki con un script de solo lectura. La bitácora completa, con la salida íntegra, está en `docs/bitacora_exploracion_fase1.md` del repositorio.

**Hallazgos.**

| | WWI | WWII | Guerra Fría |
|---|---|---|---|
| Categoría | `Kategorio:Unua Mondmilito` | `Kategorio:Dua Mondmilito` | `Kategorio:Malvarma milito` |
| Páginas directas | 16 | 134 | 57 |
| Subcategorías | 13 | 27 | 17 |
| Muestra analizada | 16 | 20 | 20 |
| Palabras mín / mediana / máx | 33 / 305 / 7851 | 0 / 212 / 1557 | 61 / 280 / 6294 |
| Artículos vacíos | 0 | 1 | 0 |
| Párrafos (≥10 palabras) por artículo | ≈ 19,6 | ≈ 5,3 | ≈ 13,6 |
| Orden de magnitud de párrafos (páginas directas)* | ≈ 310 | ≈ 710 | ≈ 775 |

\* Estimación grosera (páginas directas × media de la muestra). Las muestras de WWII y Guerra Fría no son aleatorias y unos pocos artículos muy largos inflan la media.

**Solapes entre páginas directas.** WWI ∩ WWII = 5 (el 31 % de las páginas de WWI), WWI ∩ Guerra Fría = 0 y WWII ∩ Guerra Fría = 2. "Sekvoj de la Dua Mondmilito" es subcategoría tanto de WWII como de Guerra Fría.

**Valoración.** Es viable y las categorías son semánticamente cercanas, como pide el enunciado. WWI es la clase más pequeña (≈ 300 párrafos solo con páginas directas), por lo que se estudia incluir subcategorías y equilibrar las clases. El solape implica ruido en las etiquetas: descartaremos páginas presentes en más de una clase y mediremos el ruido residual.

**Incidencias resueltas.** Los nombres de categoría distinguen mayúsculas; la API de extractos solo devuelve un texto completo por petición; y hubo que gestionar el error 429 (demasiadas peticiones) con reintentos y pausas.

**Siguientes pasos.** Medir el volumen al incluir subcategorías, fijar las reglas de limpieza y construir el generador del dataset (CSV `text;class`). Las categorías ya han sido validadas por el profesor.

### Estudio de profundidad de subcategorías

**Objetivo.** La exploración mostró que buena parte del contenido de las tres clases está en subcategorías. Medimos (28/10/2026) cuántos párrafos únicos se obtienen al bajar 0, 1 o 2 niveles en el árbol de categorías de cada clase. Se descargaron los 1.366 artículos distintos hasta profundidad 2, así que es un recuento completo, no una muestra. El detalle está en `docs/estudio_profundidad_fase1.md` del repositorio.

**Párrafos únicos por clase** (líneas de 10 palabras o más, sin contar los artículos que comparten varias clases):

| Profundidad | WWI | WWII | Guerra Fría |
|---|---:|---:|---:|
| 0 (páginas directas) | 269 | 3.231 | 610 |
| 1 | 875 | 5.103 | 1.740 |
| 2 | 1.246 | 11.316 | 2.412 |

**Solapes.** Artículos presentes en dos clases a profundidad 1: WWII ∩ Guerra Fría = 21, WWI ∩ WWII = 12 y WWI ∩ Guerra Fría = 1. El par más confundible es WWII ↔ Guerra Fría. Son temas de la frontera 1943–1949, como Yalta, Potsdam o el Telón de Acero, y casi todos entran por la subcategoría "Sekvoj de la Dua Mondmilito", que cuelga de ambas raíces. Le sigue WWI ↔ WWII, por el periodo de entreguerras y por los militares que participaron en las dos guerras.

**Decisiones.**
1. **Profundidad 1.** Es la mínima con la que WWI supera con holgura los 300 párrafos (875). A profundidad 2, WWII se multiplica por 2,2 y el solape WWII ∩ Guerra Fría sube de 21 a 50 artículos.
2. **Excluir dos subcategorías:** "Sekvoj de la Dua Mondmilito", porque es compartida, y "Zairo", porque su relación con el tema es indirecta. Las subcategorías dudosas (biografías, lugares, obras) se mantienen y su ruido se mide más adelante.
3. **Descartar los artículos presentes en más de una clase.** Es más limpio y defendible que asignarlos a una sola.
4. **Equilibrar las clases** recortando cada una al tamaño de la menor, con muestreo aleatorio y semilla fija. En la Fase 3 se compara con el dataset sin equilibrar.
5. **Conservar números y fechas** por ahora. Se decide en la Fase 2, porque son una pista muy fuerte (1914–18, 1939–45, 1947–91).

### Generación del dataset

El código de generación está en el cuaderno, como pide el enunciado, pero se puede desactivar con `GENERAR_DATASET`. Con `False`, el cuaderno carga el CSV ya generado desde el repositorio. Con `True`, lo vuelve a construir desde la API de Vikipedio, lo que tarda varios minutos por las pausas entre peticiones.

In [1]:
# --- Importaciones ---
import re
import time
import random
from collections import deque

import requests
import pandas as pd

# --- Parámetros de ejecución ---
GENERAR_DATASET = True   # True: construir el dataset desde la API · False: cargar el CSV del repo

# --- Fuente y categorías (decisiones del estudio de profundidad) ---
API_URL = "https://eo.wikipedia.org/w/api.php"
CATEGORIAS = {                                   # clase -> categoría raíz (distingue mayúsculas)
    "WWI": "Kategorio:Unua Mondmilito",
    "WWII": "Kategorio:Dua Mondmilito",
    "Guerra Fría": "Kategorio:Malvarma milito",
}
PROFUNDIDAD = 1                                  # decisión 1
SUBCATS_EXCLUIDAS = {                            # decisión 2
    "Kategorio:Sekvoj de la Dua Mondmilito",
    "Kategorio:Zairo",
}

# --- Limpieza y equilibrado ---
MIN_PALABRAS = 10                                # párrafo = línea con 10 palabras o más
SEMILLA = 42                                     # decisión 4 (reproducibilidad)

# --- Cortesía con la API (evita el error 429) ---
PAUSA = 1.0                                      # segundos entre peticiones
MAX_REINTENTOS = 4
USER_AGENT = "ClasificadorMilito/0.1 (https://github.com/Freibott/clasificador-milito; práctica CUNEF)"

# --- Rutas ---
RUTA_CSV = "data/dataset.csv"
URL_CSV_REMOTO = "https://raw.githubusercontent.com/Freibott/clasificador-milito/main/data/dataset.csv"

In [2]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})  # Wikimedia exige identificarse


def consultar_api(params):
    """Hace una petición GET a la API de MediaWiki y devuelve el JSON como diccionario.

    Centraliza la identificación (User-Agent), las pausas entre peticiones y los
    reintentos con espera creciente ante el límite de peticiones (error 429).
    """
    p = dict(params)                 # copia: no modificamos el diccionario de quien llama
    # Establecemos formato json
    p["format"] = "json"
    p["formatversion"] = 2           # MediaWiki: devuelve listas en vez de diccionarios por ID

    for intento in range(MAX_REINTENTOS):
        respuesta = session.get(API_URL, params=p, timeout=30)

        # 429 -> demasiadas peticiones, 5xx -> error del servidor: se reintenta
        if respuesta.status_code == 429 or respuesta.status_code >= 500:
            time.sleep(PAUSA * 2**intento)   # espera creciente (1, 2, 4, 8 s)
            continue

        respuesta.raise_for_status()         # otros errores HTTP (400, 403, 404): no se reintentan
        datos = respuesta.json()

        # MediaWiki responde 200 incluso con errores de la consulta
        if "error" in datos:
            raise RuntimeError(f"Error en la API de MediaWiki: {datos['error']['info']}")

        time.sleep(PAUSA)                    # pausa para no saturar el servidor
        return datos

    # Solo se llega aquí si se agotan todos los intentos
    raise RuntimeError(f"La API falló tras {MAX_REINTENTOS} intentos, último código: {respuesta.status_code}")

Comprobación: la raíz de WWI debe devolver 29 miembros directos (16 páginas y 13 subcategorías).

In [3]:
datos = consultar_api({
    "action": "query",
    "list": "categorymembers",
    "cmtitle": CATEGORIAS["WWI"],
    "cmlimit": 500,
})
len(datos["query"]["categorymembers"])  # → 29

29

## Fase 2 · NLP

## Fase 3 · Clasificación